<a href="https://colab.research.google.com/github/yash-gt08/C-Plus-Plus/blob/master/restraunt_review_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing libraries

In [2]:
import pandas as pd
import numpy as np
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


Loading the dataset

In [3]:
df = pd.read_csv("Restaurant_Reviews.tsv", delimiter="\t", quoting=3)


Inspect the dataset

In [4]:
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


STEP 2: Text Cleaning (VERY IMPORTANT)

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = text.split()
    text = " ".join(text)
    return text


Apply cleaning:

In [6]:
df["clean_review"] = df["Review"].apply(clean_text)


STEP 3: Prepare Inputs & Labels

In [7]:
reviews = df["clean_review"].tolist()
labels = df["Liked"].values


STEP 4: Tokenization (Text → Numbers)

In [8]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(reviews)

sequences = tokenizer.texts_to_sequences(reviews)


STEP 5: Padding (Fix Sequence Length)

In [9]:
max_len = 50
padded_sequences = pad_sequences(
    sequences,
    maxlen=max_len,
    padding="post",
    truncating="post"
)


STEP 6: Train–Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    labels,
    test_size=0.2,
    random_state=42
)


STEP 7: Build the LSTM Model

In [11]:
model = Sequential()

model.add(
    Embedding(
        input_dim=10000,
        output_dim=128,
        input_length=max_len
    )
)

model.add(
    Bidirectional(
        LSTM(64, return_sequences=False)
    )
)

model.add(Dropout(0.5))

model.add(Dense(1, activation="sigmoid"))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


STEP 8: Compile the Model

In [12]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


STEP 9: Train the Model

In [13]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)


Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - accuracy: 0.5153 - loss: 0.6913 - val_accuracy: 0.4750 - val_loss: 0.6931
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.6181 - loss: 0.6689 - val_accuracy: 0.6250 - val_loss: 0.6760
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 173ms/step - accuracy: 0.7903 - loss: 0.5535 - val_accuracy: 0.6750 - val_loss: 0.5544
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.8931 - loss: 0.2970 - val_accuracy: 0.8000 - val_loss: 0.4332
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 0.9736 - loss: 0.1227 - val_accuracy: 0.8125 - val_loss: 0.3842
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.9875 - loss: 0.0705 - val_accuracy: 0.8125 - val_loss: 0.4157
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 0.9931 - loss: 0.0476 - val_accuracy: 0.8500 - val_loss: 0.4501
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 104ms/step - accuracy: 0.9903 - loss: 0.0360 - val_accuracy: 0.8250 -

STEP 10: Evaluate the Model

In [14]:
y_pred = (model.predict(X_test) > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step
Accuracy: 0.775
              precision    recall  f1-score   support

           0       0.70      0.92      0.80        96
           1       0.89      0.64      0.75       104

    accuracy                           0.78       200
   macro avg       0.80      0.78      0.77       200
weighted avg       0.80      0.78      0.77       200



STEP 11: Test with Custom Input

In [15]:
def predict_review(review):
    review = clean_text(review)
    seq = tokenizer.texts_to_sequences([review])
    pad = pad_sequences(seq, maxlen=max_len, padding="post")
    pred = model.predict(pad)[0][0]
    return "Positive" if pred > 0.5 else "Negative"


In [ ]:
predict_review("The food was amazing but service was slow")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'Negative'

In [ ]:
predict_review("The food was amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


'Positive'

In [ ]:
predict_review("The food was too tasty than i thought")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


'Positive'

In [ ]:
predict_review("The food was not tasty and smelled fouled")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


'Negative'

In [ ]:
predict_review("The restraunt owner was very rude")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'Negative'